# Run: fault tests

Replay games, inject faults from `faults.yaml` at scheduled plies, run perception + game logic, and
record whether each fault was handled (flagged / correct board / no robot action on a wrong board).

**TODO**
- [ ] finish `inject` / `undo` in fault_injector and capture in data_generator
- [ ] hook up the board pipeline (gambit_vision) and the game's move validation (gambit_game)
- [ ] pass criteria per fault type (see `expected` in faults.yaml); results table -> results/sim_faults/

In [ ]:
%run ./fault_injector.ipynb
import pandas as pd

PGN = "1. e4 e5 2. Nf3 Nc6 3. Bb5 a6 4. Ba4 Nf6 5. O-O Be7 6. Re1 b5 7. Bb3 d6 8. c3 O-O *"
backend = GzCliBackend()
rng = np.random.default_rng(FAULTS["seed"])

In [ ]:
def observe_board():
    """TODO: grab_frame() -> gambit_vision board prediction (fen, low_confidence, legal)."""
    raise NotImplementedError("TODO: observe_board")

results = []
plies = pgn_replay(PGN)
schedule = set(fault_schedule(len(plies) - 1, rng))
tracker = PieceTracker.from_fen()
backend.send_all(plies[0].requests)
for ply in plies[1:]:
    backend.send_all(tracker.apply_move(ply.uci))
    if ply.idx not in schedule:
        continue
    fault = sample_fault(tracker, rng)
    inject(fault, backend)
    pred = observe_board()
    results.append({"ply": ply.idx, "kind": fault.kind, "desc": fault.description,
                    "expected_fen": fault.expected_fen or tracker.board.board_fen(), **pred})
    undo(fault, tracker, backend)
pd.DataFrame(results)